<a href="https://colab.research.google.com/github/labdall4/CPS_Project/blob/main/RPS_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Evaluation File**

Cell 1: Header and Shared Spec Constants

In [1]:
# RPS-1: Evaluation Notebook
# Owner: evaluation. Follows SPEC.ipynb v1.1.
# Evaluates RPS_detection (detect_hands) and RPS_strategy (decide) on demo/test splits
# and verifies end-to-end expected payoffs.

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Copied from SPEC.ipynb Section 1. Do not edit these here; change them in the spec.
SHAPES = ['Rock', 'Paper', 'Scissors']
PLAYERS = ['self', 'opponent']
NUM_HANDS = 4
SEED = 42
PAYOFF = {'win': 1, 'tie': 0, 'loss': -1}
BEATS = {'Rock': 'Scissors', 'Scissors': 'Paper', 'Paper': 'Rock'}

Cell 2: Import Contracts from SPEC, RPS_detection, and RPS_strategy

In [2]:
# Load sibling notebooks so detect_hands(), check_hands(), and decide() are available.
# If RPS_detection.ipynb is still in progress, a clear message is printed and synthetic
# checks can still run.

try:
    %run -i RPS_strategy.ipynb
    print('Loaded RPS_strategy.ipynb successfully.')
except Exception as e:
    print('Warning: could not run RPS_strategy.ipynb:', e)

try:
    %run -i RPS_detection.ipynb
    print('Loaded RPS_detection.ipynb successfully.')
except Exception as e:
    print('Warning: RPS_detection.ipynb not ready yet:', e)

# Ensure check_hands() from SPEC Section 3 is defined even if SPEC.ipynb was not %run
if 'check_hands' not in globals():
    def check_hands(hands):
        '''Validate a list of hand records against Section 2. Raises ValueError on any violation. Returns True otherwise.'''
        if len(hands) != NUM_HANDS:
            raise ValueError('expected 4 hands, got ' + str(len(hands)))
        if sorted(h['position'] for h in hands) != [0, 1, 2, 3]:
            raise ValueError('positions must be 0 to 3, each used once')
        for h in hands:
            if h['player'] not in PLAYERS:
                raise ValueError('bad player: ' + str(h['player']))
            if h['shape'] not in SHAPES:
                raise ValueError('bad shape: ' + str(h['shape']))
        counts = {p: sum(1 for h in hands if h['player'] == p) for p in PLAYERS}
        if counts['self'] != 2 or counts['opponent'] != 2:
            raise ValueError('need 2 hands per player, got ' + str(counts))
        return True

Cell 3: Detection Evaluation Against data/expected_labels.json

In [ ]:
def evaluate_detection(image_dir='data/demo', labels_path='data/expected_labels.json'):
    """
    Evaluate detect_hands() on every labeled image in image_dir.
    Returns a dictionary of metrics and a 3x3 shape confusion matrix (rows=expected, cols=predicted).
    """
    labels_file = Path(labels_path)
    if not labels_file.exists():
        raise FileNotFoundError(f'Expected labels file not found: {labels_file}')

    with open(labels_file, 'r') as f:
        expected_map = json.load(f)

    img_dir = Path(image_dir)
    image_files = sorted([
        p for p in img_dir.iterdir()
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.name in expected_map
    ])

    if not image_files:
        print(f'No matching labeled images found in {image_dir}.')
        return None

    shape_to_idx = {s: i for i, s in enumerate(SHAPES)}
    confusion = np.zeros((3, 3), dtype=int)

    total_images = len(image_files)
    valid_images = 0
    exact_images = 0
    total_hands = 0
    player_correct = 0
    shape_correct = 0
    failures = []

    for img_path in image_files:
        expected_hands = sorted(expected_map[img_path.name], key=lambda h: h['position'])
        try:
            pred_hands = detect_hands(str(img_path))
            check_hands(pred_hands)
            valid_images += 1
        except Exception as exc:
            failures.append((img_path.name, str(exc)))
            total_hands += NUM_HANDS
            continue

        all_four_correct = True
        for pred_h, exp_h in zip(pred_hands, expected_hands):
            total_hands += 1
            p_ok = (pred_h['player'] == exp_h['player'])
            s_ok = (pred_h['shape'] == exp_h['shape'])

            if p_ok:
                player_correct += 1
            if s_ok:
                shape_correct += 1
            if not (p_ok and s_ok):
                all_four_correct = False

            confusion[shape_to_idx[exp_h['shape']], shape_to_idx[pred_h['shape']]] += 1

        if all_four_correct:
            exact_images += 1

    metrics = {
        'split': image_dir,
        'total_images': total_images,
        'valid_contract_images': valid_images,
        'exact_match_images': exact_images,
        'player_accuracy': player_correct / max(total_hands, 1),
        'shape_accuracy': shape_correct / max(total_hands, 1),
        'failures': failures,
        'confusion_matrix': confusion,
    }
    return metrics

# Run on demo split (data/demo/demo_01.jpg .. demo_05.jpg)
if Path('data/demo').exists() and Path('data/expected_labels.json').exists() and 'detect_hands' in globals():
    demo_results = evaluate_detection('data/demo', 'data/expected_labels.json')
    if demo_results:
        print(f"Split: {demo_results['split']}")
        print(f"Valid 4-hand detections: {demo_results['valid_contract_images']}/{demo_results['total_images']}")
        print(f"Exact 4-hand image matches: {demo_results['exact_match_images']}/{demo_results['total_images']}")
        print(f"Player accuracy (self/opponent): {demo_results['player_accuracy']:.1%}")
        print(f"Shape accuracy (R/P/S): {demo_results['shape_accuracy']:.1%}")
        if demo_results['failures']:
            print('Failures:', demo_results['failures'])
else:
    print('Skipping demo image evaluation (waiting for data/demo, expected_labels.json, or detect_hands).')

Cell 4: End-to-End Integration and Payoff Simulation

In [ ]:
def round_score(self_shape, opp_shape):
    if self_shape == opp_shape:
        return PAYOFF['tie']
    return PAYOFF['win'] if BEATS[self_shape] == opp_shape else PAYOFF['loss']

def evaluate_strategy_payoffs(n_rounds=10000, seed=SEED):
    """
    Verify empirical payoff convergence of decide() across key Stage 1 matchups
    against both an optimal (Nash) opponent and a naive 'never-lose' opponent.
    """
    rng = np.random.default_rng(seed)

    # Worked example from SPEC Section 6: Self=[Paper, Scissors], Opponent=[Paper, Rock]
    self_hands = [
        {'player': 'self', 'shape': 'Paper', 'position': 0, 'confidence': 0.95},
        {'player': 'self', 'shape': 'Scissors', 'position': 1, 'confidence': 0.90},
    ]
    opp_hands = [
        {'player': 'opponent', 'shape': 'Paper', 'position': 2, 'confidence': 0.88},
        {'player': 'opponent', 'shape': 'Rock', 'position': 3, 'confidence': 0.93},
    ]

    # Case A: Opponent also plays Nash equilibrium (Paper with 2/3, Rock with 1/3)
    nash_scores = []
    for _ in range(n_rounds):
        action = decide(self_hands, opp_hands, rng=rng)
        opp_shape = 'Paper' if rng.random() < (2 / 3) else 'Rock'
        nash_scores.append(round_score(action['keep_shape'], opp_shape))

    # Case B: Opponent plays naive "never lose" strategy (always keeps Paper)
    naive_scores = []
    for _ in range(n_rounds):
        action = decide(self_hands, opp_hands, rng=rng)
        naive_scores.append(round_score(action['keep_shape'], 'Paper'))

    print(f"PS vs PR over {n_rounds} rounds:")
    print(f"  vs. Optimal Opponent  -> Empirical EV: {np.mean(nash_scores):+.4f} (Expected: +0.3333)")
    print(f"  vs. Always-Paper Opp  -> Empirical EV: {np.mean(naive_scores):+.4f} (Expected: +0.3333)")

    assert abs(np.mean(nash_scores) - (1 / 3)) < 0.03
    assert abs(np.mean(naive_scores) - (1 / 3)) < 0.03
    return nash_scores, naive_scores

nash_scores, naive_scores = evaluate_strategy_payoffs()

Cell 5: Plots (Cumulative Average Payoff and Confusion Matrix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Running average payoff convergence
rounds = np.arange(1, len(nash_scores) + 1)
axes[0].plot(rounds, np.cumsum(nash_scores) / rounds, label='vs. Nash Opponent')
axes[0].plot(rounds, np.cumsum(naive_scores) / rounds, label='vs. Always-Paper Opponent', alpha=0.8)
axes[0].axhline(1 / 3, color='black', linestyle='--', label='Theoretical EV (1/3)')
axes[0].set_title('Stage 2 Empirical Payoff Convergence (PS vs PR)')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Cumulative Average Payoff')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Shape Confusion Matrix (uses demo_results if available, else placeholder)
if 'demo_results' in globals() and demo_results is not None:
    cm = demo_results['confusion_matrix']
else:
    cm = np.zeros((3, 3), dtype=int)

im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks(range(3))
axes[1].set_yticks(range(3))
axes[1].set_xticklabels(SHAPES)
axes[1].set_yticklabels(SHAPES)
axes[1].set_xlabel('Predicted Shape')
axes[1].set_ylabel('Expected Shape')
axes[1].set_title('Detection Confusion Matrix')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

Cell 6: Separate Test Set Evaluation & Environment Freeze (SPEC Sections 4 & 5)

In [ ]:
# Section 4: Evaluate on data/test/ ONLY after detection/tuning is finalized.
if Path('data/test').exists() and Path('data/expected_labels.json').exists() and 'detect_hands' in globals():
    test_results = evaluate_detection('data/test', 'data/expected_labels.json')
    if test_results:
        print('Test split results:', {k: v for k, v in test_results.items() if k != 'confusion_matrix'})

# Section 5: Record pinned package versions after first working install
!pip freeze